# HW08-09 – PyTorch MLP: регуляризация и оптимизация

Требования см. `homeworks/HW08-09/TS/S08-homework.md`.

Ноутбук запускает эксперименты E1–E4 и O1–O3 и генерирует артефакты в `./artifacts/`.

In [1]:
from pathlib import Path

import numpy as np
import torch
import torchvision
import matplotlib.pyplot as plt

print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)

ARTIFACTS_DIR = Path("artifacts")
FIGURES_DIR = ARTIFACTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)



torch: 2.10.0+cpu
torchvision: 0.25.0+cpu


In [2]:
from __future__ import annotations

import json
import math
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple


def _require_torch() -> None:
    try:
        import torch
        import torchvision
    except ModuleNotFoundError as e:
        raise ModuleNotFoundError(
            "PyTorch is not installed in this environment. "
            "Install torch + torchvision and rerun the notebook."
        ) from e


def set_seed(seed: int) -> None:
    _require_torch()
    import numpy as np
    import torch

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_device() -> "torch.device":
    _require_torch()
    import torch

    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def accuracy_from_logits(logits: "torch.Tensor", y: "torch.Tensor") -> float:
    _require_torch()
    import torch

    preds = torch.argmax(logits, dim=1)
    correct = (preds == y).sum().item()
    return correct / max(1, y.numel())


def _dataset_transform(dataset: str) -> "torchvision.transforms.Compose":
    _require_torch()
    import torchvision.transforms as T

    dataset = dataset.upper()
    if dataset != "CIFAR10":
        raise ValueError(
            "This homework is configured for variant C (CIFAR10). "
            "Set DATASET='CIFAR10' and restart the kernel, then Run All."
        )
    return T.Compose(
        [
            T.ToTensor(),
            T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
        ]
    )


def load_datasets(
    dataset: str,
    data_root: Path,
) -> Tuple["torch.utils.data.Dataset", "torch.utils.data.Dataset"]:
    _require_torch()
    from torchvision import datasets

    dataset = dataset.upper()
    transform = _dataset_transform(dataset)

    train = datasets.CIFAR10(root=str(data_root), train=True, download=True, transform=transform)
    test = datasets.CIFAR10(root=str(data_root), train=False, download=True, transform=transform)
    return train, test


def split_train_val(
    train_dataset: "torch.utils.data.Dataset",
    val_fraction: float,
    seed: int,
) -> Tuple["torch.utils.data.Dataset", "torch.utils.data.Dataset"]:
    _require_torch()
    import torch

    if not (0.0 < val_fraction < 1.0):
        raise ValueError("val_fraction must be in (0, 1)")

    n_total = len(train_dataset)
    n_val = int(math.floor(n_total * val_fraction))
    n_train = n_total - n_val
    generator = torch.Generator().manual_seed(seed)
    train_split, val_split = torch.utils.data.random_split(
        train_dataset, [n_train, n_val], generator=generator
    )
    return train_split, val_split


def make_loaders(
    train_ds: "torch.utils.data.Dataset",
    val_ds: "torch.utils.data.Dataset",
    test_ds: "torch.utils.data.Dataset",
    batch_size: int,
    num_workers: int = 0,
) -> Tuple["torch.utils.data.DataLoader", "torch.utils.data.DataLoader", "torch.utils.data.DataLoader"]:
    _require_torch()
    import torch

    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers
    )
    val_loader = torch.utils.data.DataLoader(
        val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers
    )
    test_loader = torch.utils.data.DataLoader(
        test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers
    )
    return train_loader, val_loader, test_loader


_require_torch()
import torch.nn as nn


class MLP(nn.Module):
    def __init__(
        self,
        *,
        input_dim: int,
        num_classes: int,
        hidden_sizes: List[int],
        dropout_p: float = 0.0,
        use_batchnorm: bool = False,
    ) -> None:
        super().__init__()

        if not hidden_sizes:
            raise ValueError("hidden_sizes must be non-empty")
        if dropout_p < 0.0 or dropout_p >= 1.0:
            raise ValueError("dropout_p must be in [0, 1)")

        layers: List[nn.Module] = [nn.Flatten()]
        prev = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout_p > 0.0:
                layers.append(nn.Dropout(p=dropout_p))
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def build_mlp_classifier(
    *,
    input_dim: int,
    num_classes: int,
    hidden_sizes: List[int],
    dropout_p: float = 0.0,
    use_batchnorm: bool = False,
) -> "torch.nn.Module":
    _require_torch()
    return MLP(
        input_dim=input_dim,
        num_classes=num_classes,
        hidden_sizes=hidden_sizes,
        dropout_p=dropout_p,
        use_batchnorm=use_batchnorm,
    )


@dataclass(frozen=True)
class EarlyStoppingConfig:
    patience: int = 4
    monitor: str = "val_accuracy"
    mode: str = "max"


class EarlyStopper:
    def __init__(self, cfg: EarlyStoppingConfig) -> None:
        if cfg.patience < 1:
            raise ValueError("patience must be >= 1")
        if cfg.mode not in {"min", "max"}:
            raise ValueError("mode must be 'min' or 'max'")
        if cfg.monitor not in {"val_accuracy", "val_loss"}:
            raise ValueError("monitor must be 'val_accuracy' or 'val_loss'")
        self.cfg = cfg
        self.best_score: Optional[float] = None
        self.best_state: Optional[Dict[str, Any]] = None
        self.bad_epochs = 0

    def _is_improvement(self, score: float) -> bool:
        if self.best_score is None:
            return True
        if self.cfg.mode == "max":
            return score > self.best_score
        return score < self.best_score

    def step(self, score: float, model: "torch.nn.Module") -> bool:
        _require_torch()

        if self._is_improvement(score):
            self.best_score = score
            self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            self.bad_epochs = 0
            return False

        self.bad_epochs += 1
        return self.bad_epochs >= self.cfg.patience


def train_one_epoch(
    model: "torch.nn.Module",
    loader: "torch.utils.data.DataLoader",
    loss_fn: "torch.nn.Module",
    optimizer: "torch.optim.Optimizer",
    device: "torch.device",
) -> Tuple[float, float]:
    _require_torch()
    import torch

    model.train()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()

        batch_size = y.numel()
        total_loss += loss.item() * batch_size
        total_correct += (torch.argmax(logits, dim=1) == y).sum().item()
        total_count += batch_size

    avg_loss = total_loss / max(1, total_count)
    avg_acc = total_correct / max(1, total_count)
    return avg_loss, avg_acc


def evaluate(
    model: "torch.nn.Module",
    loader: "torch.utils.data.DataLoader",
    loss_fn: "torch.nn.Module",
    device: "torch.device",
) -> Tuple[float, float]:
    _require_torch()
    import torch

    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            loss = loss_fn(logits, y)

            batch_size = y.numel()
            total_loss += loss.item() * batch_size
            total_correct += (torch.argmax(logits, dim=1) == y).sum().item()
            total_count += batch_size

    avg_loss = total_loss / max(1, total_count)
    avg_acc = total_correct / max(1, total_count)
    return avg_loss, avg_acc


def model_summary(model_cfg: Dict[str, Any]) -> str:
    hidden = model_cfg["hidden_sizes"]
    activation = model_cfg.get("activation", "ReLU")
    dropout_p = float(model_cfg.get("dropout_p", 0.0))
    use_bn = bool(model_cfg.get("use_batchnorm", False))
    return f"h={hidden} act={activation} dropout={dropout_p} bn={int(use_bn)}"


def ensure_artifacts(artifacts_dir: Path) -> Path:
    artifacts_dir.mkdir(parents=True, exist_ok=True)
    (artifacts_dir / "figures").mkdir(parents=True, exist_ok=True)
    return artifacts_dir


def _write_json(path: Path, obj: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


def _append_csv_row(path: Path, fieldnames: List[str], row: Dict[str, Any]) -> None:
    import csv

    path.parent.mkdir(parents=True, exist_ok=True)
    is_new = not path.exists()
    with path.open("a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if is_new:
            writer.writeheader()
        writer.writerow({k: row.get(k, "") for k in fieldnames})


def _plot_curves(
    history: Dict[str, List[float]],
    title: str,
    out_path: Path,
) -> None:
    import matplotlib.pyplot as plt

    out_path.parent.mkdir(parents=True, exist_ok=True)
    epochs = list(range(1, len(history["train_loss"]) + 1))

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(epochs, history["train_loss"], label="train")
    ax[0].plot(epochs, history["val_loss"], label="val")
    ax[0].set_title("Loss")
    ax[0].set_xlabel("epoch")
    ax[0].grid(True, alpha=0.25)
    ax[0].legend()

    ax[1].plot(epochs, history["train_accuracy"], label="train")
    ax[1].plot(epochs, history["val_accuracy"], label="val")
    ax[1].set_title("Accuracy")
    ax[1].set_xlabel("epoch")
    ax[1].grid(True, alpha=0.25)
    ax[1].legend()

    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def _plot_lr_extremes(
    o1: Dict[str, List[float]],
    o2: Dict[str, List[float]],
    out_path: Path,
) -> None:
    import matplotlib.pyplot as plt

    out_path.parent.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for a, hist, name in [
        (ax[0], o1, "O1: LR too large"),
        (ax[1], o2, "O2: LR too small"),
    ]:
        epochs = list(range(1, len(hist["train_loss"]) + 1))
        a.plot(epochs, hist["train_loss"], label="train loss")
        a.plot(epochs, hist["val_loss"], label="val loss")
        a.set_title(name)
        a.set_xlabel("epoch")
        a.grid(True, alpha=0.25)
        a.legend()

    fig.suptitle("LR extremes diagnostics")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def run_training(
    experiment_id: str,
    dataset: str,
    seed: int,
    model_cfg: Dict[str, Any],
    train_cfg: Dict[str, Any],
    train_loader: "torch.utils.data.DataLoader",
    val_loader: "torch.utils.data.DataLoader",
    device: "torch.device",
    early_stopping: Optional[EarlyStoppingConfig] = None,
) -> Tuple[Dict[str, Any], Dict[str, List[float]], Dict[str, Any]]:
    """
    Returns (result_row, history, best_state_dict_cpu).
    For early-stopping runs this is the best state; otherwise it's the final state.
    """
    _require_torch()
    import torch

    input_dim = int(model_cfg["input_dim"])
    num_classes = int(model_cfg["num_classes"])

    model = build_mlp_classifier(
        input_dim=input_dim,
        num_classes=num_classes,
        hidden_sizes=list(model_cfg["hidden_sizes"]),
        dropout_p=float(model_cfg.get("dropout_p", 0.0)),
        use_batchnorm=bool(model_cfg.get("use_batchnorm", False)),
    ).to(device)

    loss_fn = torch.nn.CrossEntropyLoss()

    optimizer_name = str(train_cfg.get("optimizer", "Adam"))
    lr = float(train_cfg["lr"])
    weight_decay = float(train_cfg.get("weight_decay", 0.0))
    momentum = float(train_cfg.get("momentum", 0.0))

    if optimizer_name.lower() == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name.lower() == "sgd":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=momentum,
            weight_decay=weight_decay,
        )
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    max_epochs = int(train_cfg["epochs"])

    history: Dict[str, List[float]] = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
    }

    stopper = EarlyStopper(early_stopping) if early_stopping else None

    best_val_acc = -1.0
    best_val_loss = float("inf")
    best_state: Optional[Dict[str, Any]] = None

    for epoch in range(1, max_epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)

        history["train_loss"].append(float(train_loss))
        history["train_accuracy"].append(float(train_acc))
        history["val_loss"].append(float(val_loss))
        history["val_accuracy"].append(float(val_acc))

        if val_acc > best_val_acc:
            best_val_acc = float(val_acc)
        if val_loss < best_val_loss:
            best_val_loss = float(val_loss)

        if stopper:
            score = val_acc if stopper.cfg.monitor == "val_accuracy" else val_loss
            should_stop = stopper.step(float(score), model)
            if should_stop:
                break

    if stopper and stopper.best_state:
        best_state = stopper.best_state
    else:
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    assert best_state is not None
    epochs_trained = len(history["train_loss"])
    result_row: Dict[str, Any] = {
        "experiment_id": experiment_id,
        "dataset": dataset,
        "seed": seed,
        "model_summary": model_summary(model_cfg),
        "optimizer": "Adam" if optimizer_name.lower() == "adam" else "SGD",
        "lr": lr,
        "momentum": momentum if optimizer_name.lower() == "sgd" else 0.0,
        "weight_decay": weight_decay,
        "epochs_trained": epochs_trained,
        "best_val_accuracy": best_val_acc,
        "best_val_loss": best_val_loss,
    }
    return result_row, history, best_state


def run_all_experiments(
    *,
    dataset: str,
    seed: int,
    fast_mode: bool = False,
    data_root: Path,
    artifacts_dir: Path,
    val_fraction: float = 0.2,
    batch_size: int = 256,
    num_workers: int = 0,
    hidden_sizes: Optional[List[int]] = None,
    base_lr: float = 1e-3,
    base_epochs: int = 15,
    dropout_p: float = 0.3,
    early_patience: int = 4,
) -> Dict[str, Any]:
    _require_torch()
    import torch

    artifacts_dir = ensure_artifacts(artifacts_dir)
    figures_dir = artifacts_dir / "figures"

    set_seed(seed)
    device = get_device()

    train_full, test_ds = load_datasets(dataset, data_root=data_root)
    train_ds, val_ds = split_train_val(train_full, val_fraction=val_fraction, seed=seed)
    train_loader, val_loader, test_loader = make_loaders(
        train_ds,
        val_ds,
        test_ds,
        batch_size=batch_size,
        num_workers=num_workers,
    )

    x0, y0 = next(iter(train_loader))
    input_dim = int(x0[0].numel())
    if hasattr(train_full, "classes"):
        num_classes = int(len(train_full.classes))
    elif hasattr(train_full, "targets"):
        targets = train_full.targets
        if isinstance(targets, torch.Tensor):
            num_classes = int(torch.unique(targets).numel())
        else:
            num_classes = int(len(set(int(t) for t in targets)))
    else:
        num_classes = int(torch.max(y0).item() + 1)

    if hidden_sizes is None:
        hidden_sizes = [512, 256, 128]

    base_model = {
        "input_dim": input_dim,
        "num_classes": num_classes,
        "hidden_sizes": hidden_sizes,
        "activation": "ReLU",
        "dropout_p": 0.0,
        "use_batchnorm": False,
    }

    runs_csv = artifacts_dir / "runs.csv"
    if runs_csv.exists():
        runs_csv.unlink()
    fields = [
        "experiment_id",
        "dataset",
        "seed",
        "model_summary",
        "optimizer",
        "lr",
        "momentum",
        "weight_decay",
        "epochs_trained",
        "best_val_accuracy",
        "best_val_loss",
    ]

    histories: Dict[str, Dict[str, List[float]]] = {}

    e1_row, e1_hist, _ = run_training(
        experiment_id="E1",
        dataset=dataset,
        seed=seed,
        model_cfg=base_model,
        train_cfg={"optimizer": "Adam", "lr": base_lr, "epochs": base_epochs, "weight_decay": 0.0},
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        early_stopping=None,
    )
    histories["E1"] = e1_hist
    _append_csv_row(runs_csv, fields, e1_row)

    e2_model = dict(base_model)
    e2_model["dropout_p"] = float(dropout_p)
    e2_row, e2_hist, _ = run_training(
        experiment_id="E2",
        dataset=dataset,
        seed=seed,
        model_cfg=e2_model,
        train_cfg={"optimizer": "Adam", "lr": base_lr, "epochs": base_epochs, "weight_decay": 0.0},
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        early_stopping=None,
    )
    histories["E2"] = e2_hist
    _append_csv_row(runs_csv, fields, e2_row)

    e3_model = dict(base_model)
    e3_model["use_batchnorm"] = True
    e3_row, e3_hist, _ = run_training(
        experiment_id="E3",
        dataset=dataset,
        seed=seed,
        model_cfg=e3_model,
        train_cfg={"optimizer": "Adam", "lr": base_lr, "epochs": base_epochs, "weight_decay": 0.0},
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        early_stopping=None,
    )
    histories["E3"] = e3_hist
    _append_csv_row(runs_csv, fields, e3_row)

    best_pre = "E2" if float(e2_row["best_val_accuracy"]) >= float(e3_row["best_val_accuracy"]) else "E3"
    e4_model = e2_model if best_pre == "E2" else e3_model

    e4_row, e4_hist, e4_best_state = run_training(
        experiment_id="E4",
        dataset=dataset,
        seed=seed,
        model_cfg=e4_model,
        train_cfg={"optimizer": "Adam", "lr": base_lr, "epochs": base_epochs, "weight_decay": 0.0},
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        early_stopping=EarlyStoppingConfig(
            patience=int(early_patience),
            monitor="val_accuracy",
            mode="max",
        ),
    )
    histories["E4"] = e4_hist
    _append_csv_row(runs_csv, fields, e4_row)

    best_model_path = artifacts_dir / "best_model.pt"
    torch.save(e4_best_state, best_model_path)

    best_cfg_path = artifacts_dir / "best_config.json"
    best_cfg_base = {
        "hidden_sizes": list(e4_model["hidden_sizes"]),
        "dropout_p": float(e4_model.get("dropout_p", 0.0)),
        "use_batchnorm": bool(e4_model.get("use_batchnorm", False)),
        "optimizer": "Adam",
        "lr": float(base_lr),
        "momentum": 0.0,
        "weight_decay": 0.0,
        "dataset": dataset,
        "seed": seed,
        "fast_mode": bool(fast_mode),
        "epochs_trained": int(e4_row["epochs_trained"]),
        "best_val_accuracy": float(e4_row["best_val_accuracy"]),
        "best_val_loss": float(e4_row["best_val_loss"]),
    }

    _plot_curves(
        histories["E4"],
        title=f"E4 (best): {model_summary(e4_model)}",
        out_path=figures_dir / "curves_best.png",
    )

    o1_row, o1_hist, _ = run_training(
        experiment_id="O1",
        dataset=dataset,
        seed=seed,
        model_cfg=e4_model,
        train_cfg={"optimizer": "Adam", "lr": 1e-1, "epochs": 6, "weight_decay": 0.0},
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        early_stopping=None,
    )
    histories["O1"] = o1_hist
    _append_csv_row(runs_csv, fields, o1_row)

    o2_row, o2_hist, _ = run_training(
        experiment_id="O2",
        dataset=dataset,
        seed=seed,
        model_cfg=e4_model,
        train_cfg={"optimizer": "Adam", "lr": 1e-5, "epochs": 6, "weight_decay": 0.0},
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        early_stopping=None,
    )
    histories["O2"] = o2_hist
    _append_csv_row(runs_csv, fields, o2_row)

    o3_row, o3_hist, _ = run_training(
        experiment_id="O3",
        dataset=dataset,
        seed=seed,
        model_cfg=e4_model,
        train_cfg={
            "optimizer": "SGD",
            "lr": max(1e-4, min(1e-2, base_lr * 10)),
            "epochs": 12,
            "momentum": 0.9,
            "weight_decay": 1e-4,
        },
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        early_stopping=None,
    )
    histories["O3"] = o3_hist
    _append_csv_row(runs_csv, fields, o3_row)

    _plot_lr_extremes(histories["O1"], histories["O2"], figures_dir / "curves_lr_extremes.png")

    model_for_test = build_mlp_classifier(
        input_dim=input_dim,
        num_classes=num_classes,
        hidden_sizes=list(e4_model["hidden_sizes"]),
        dropout_p=float(e4_model.get("dropout_p", 0.0)),
        use_batchnorm=bool(e4_model.get("use_batchnorm", False)),
    ).to(device)
    state = torch.load(best_model_path, map_location=device)
    model_for_test.load_state_dict(state)

    loss_fn = torch.nn.CrossEntropyLoss()
    test_loss, test_acc = evaluate(model_for_test, test_loader, loss_fn, device)

    best_cfg = dict(best_cfg_base)
    best_cfg["test_loss"] = float(test_loss)
    best_cfg["test_accuracy"] = float(test_acc)
    _write_json(best_cfg_path, best_cfg)

    return {
        "device": str(device),
        "input_dim": input_dim,
        "num_classes": num_classes,
        "picked_from": best_pre,
        "best_val_accuracy": float(e4_row["best_val_accuracy"]),
        "best_val_loss": float(e4_row["best_val_loss"]),
        "test_accuracy": float(test_acc),
        "test_loss": float(test_loss),
        "runs_csv": os.fspath(runs_csv),
        "best_model_path": os.fspath(best_model_path),
        "best_config_path": os.fspath(best_cfg_path),
        "fig_best": os.fspath(figures_dir / "curves_best.png"),
        "fig_lr": os.fspath(figures_dir / "curves_lr_extremes.png"),
    }


In [3]:
DATASET = "CIFAR10"
SEED = 42
FAST_MODE = False

DATA_ROOT = Path("..") / "datasets"

VAL_FRACTION = 0.2
BATCH_SIZE = 256
NUM_WORKERS = 0

HIDDEN_SIZES = [512, 256, 128]
BASE_LR = 1e-3
BASE_EPOCHS = 15
DROPOUT_P = 0.3
EARLY_PATIENCE = 4


In [4]:
set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

device(type='cpu')

In [5]:
print('DATASET =', DATASET)
train_full, test_ds = load_datasets(DATASET, data_root=DATA_ROOT)
train_ds, val_ds = split_train_val(train_full, val_fraction=VAL_FRACTION, seed=SEED)
train_loader, val_loader, test_loader = make_loaders(
    train_ds,
    val_ds,
    test_ds,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

x, y = next(iter(train_loader))
print('x.shape:', tuple(x.shape), 'y.shape:', tuple(y.shape))
print('x.min/max:', float(x.min()), float(x.max()))
print('y.min/max:', int(y.min()), int(y.max()))


DATASET = CIFAR10


c:\Users\tema\nano-banana\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


x.shape: (256, 3, 32, 32) y.shape: (256,)
x.min/max: -1.9894737005233765 2.12648868560791
y.min/max: 0 9


In [6]:
summary = run_all_experiments(
    dataset=DATASET,
    seed=SEED,
    fast_mode=FAST_MODE,
    data_root=DATA_ROOT,
    artifacts_dir=ARTIFACTS_DIR,
    val_fraction=VAL_FRACTION,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    hidden_sizes=HIDDEN_SIZES,
    base_lr=BASE_LR,
    base_epochs=BASE_EPOCHS,
    dropout_p=DROPOUT_P,
    early_patience=EARLY_PATIENCE,
)
_write_json(ARTIFACTS_DIR / "summary.json", summary)
summary

{'device': 'cpu',
 'input_dim': 3072,
 'num_classes': 10,
 'picked_from': 'E3',
 'best_val_accuracy': 0.5491,
 'best_val_loss': 1.317373631286621,
 'test_accuracy': 0.5436,
 'test_loss': 1.3569832914352418,
 'runs_csv': 'artifacts\\runs.csv',
 'best_model_path': 'artifacts\\best_model.pt',
 'best_config_path': 'artifacts\\best_config.json',
 'fig_best': 'artifacts\\figures\\curves_best.png',
 'fig_lr': 'artifacts\\figures\\curves_lr_extremes.png'}

## Артефакты

- `./artifacts/runs.csv`
- `./artifacts/best_model.pt`
- `./artifacts/best_config.json`
- `./artifacts/figures/curves_best.png`
- `./artifacts/figures/curves_lr_extremes.png`
